# 擂台加賽:Gemini Flash 讀漫畫日文

用上一輪擂台存在 Drive 的 132 個裁圖(`manga-ocr/out/crops` + `results.json`),讓 Gemini Flash 逐框抄寫,再和 PaddleOCR-VL / manga-ocr / Qwen 比。

- **執行階段:CPU 就好**(不吃 GPU 單元)
- 走 Vertex AI,費用從 AI Pro 附的 $10 GCP 額度扣;132 個小請求,花費應該不到幾美分

**事前準備(一次)**
1. 到 Google Developer Program 的 My Benefits 領取 $10 額度,套到一個帳單帳戶
2. 建一個 GCP 專案,掛上那個帳單帳戶,啟用 **Vertex AI API**
3. 建議設預算警報(例如 $5)

## 1. 設定

In [ ]:
PROJECT_ID = ""          # 你的 GCP 專案 ID(必填)
LOCATION   = "global"
MODEL      = ""          # 留空 = 自動挑最新的 Gemini Flash;也可以手動指定

DRIVE_DIR  = "/content/drive/MyDrive/manga-ocr/out"
CONCURRENCY = 8

PROMPT = '''逐字抄寫圖中的日文(直書由右到左、由上到下讀)。
- 不要輸出振り仮名(漢字旁的小字注音)和傍点(字旁的強調點)
- 保留原本的標點與口語寫法,不要改寫、不要翻譯、不要解釋
- 圖中沒有文字就輸出空字串
只輸出抄寫結果。'''

## 2. 安裝、授權、掛載 Drive

In [ ]:
!pip install -q -U google-genai
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

from google import genai
from google.genai import types
assert PROJECT_ID, "先填 PROJECT_ID"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## 3. 選模型

In [ ]:
import re
names = [m.name.split("/")[-1] for m in client.models.list()]
flash = sorted(n for n in names if re.fullmatch(r"gemini-[\d.-]+-flash(-preview.*)?", n))
print("可用的 Flash:", flash)
if not MODEL:
    def ver(n):
        return [int(x) for x in re.findall(r"\d+", n.split("-flash")[0])]
    MODEL = max(flash, key=lambda n: (ver(n), "preview" not in n))
print("使用:", MODEL)

## 4. 逐框抄寫

In [ ]:
import json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

raw = json.load(open(f"{DRIVE_DIR}/results.json"))
items = []
for page, blocks in raw.items():
    pname = Path(page).stem
    for i, b in enumerate(blocks):
        items.append({"key": f"{pname}#{i+1}", **b})
print(len(items), "個框")

def ask(item):
    img = open(f"{DRIVE_DIR}/{item['crop']}", "rb").read()
    for attempt in range(5):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=[types.Part.from_bytes(data=img, mime_type="image/png"), PROMPT],
                config=types.GenerateContentConfig(temperature=0),
            )
            return (r.text or "").strip()
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                time.sleep(2 ** attempt); continue
            return f"[錯誤] {e}"
    return "[錯誤] 重試失敗"

t0 = time.time()
with ThreadPoolExecutor(CONCURRENCY) as ex:
    for item, text in zip(items, ex.map(ask, items)):
        item["gemini"] = text
print(f"完成,{time.time()-t0:.0f} 秒")
json.dump(items, open(f"{DRIVE_DIR}/results_gemini.json", "w"), ensure_ascii=False, indent=1)

## 5. 計分

- **對話**:以 PaddleOCR-VL 為基準(人工核對過,這 116 框它全對),算字元錯誤率(CER)。比對前先做正規化:全形轉半形、去掉空白換行、`……`/`．．．` 統一、長音符號統一、去掉引號和逗號、`ヵ`→`カ`、`‼` → `!!`。
- **狀聲詞**:用人工核對的正解;`プルルル` 類只看開頭是不是「プル」。
- 時鐘誤判、只有點點的框不計分。

In [ ]:
import unicodedata

SFX_TRUTH = {
    "p011#4": "シャコーン", "p018#8": "ガチ",
    "p022#1": "ポタッ", "p022#2": "ポタッ", "p022#3": "ポタッ",
    "p022#4": "ポタッ", "p022#5": "ポタッ", "p022#6": "ポタッ",
}
SFX_PREFIX = {"p017#1": "プル", "p017#2": "プル", "p017#3": "プル", "p020#1": "プル"}
SKIP = {"p008#3", "p014#1", "p014#4", "p026#5"}   # 長度難判的「ちゅうう…」、時鐘、看不清的「カッ」、純點點

def norm(s):
    s = unicodedata.normalize("NFKC", s or "")
    s = re.sub(r"\s+", "", s)
    s = s.replace("‼", "!!").replace("…", ".").replace("・", "").replace("ヵ", "カ").replace("ヶ", "ケ")
    s = re.sub(r"[、\u0022\u201c\u201d\u2018\u2019']", "", s)
    s = re.sub(r"[ー―—‐-]+", "ー", s)
    s = re.sub(r"\.+", "…", s)
    return s

def cer(ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    if not ref:
        return 0.0
    prev = list(range(len(hyp) + 1))
    for i, a in enumerate(ref, 1):
        cur = [i]
        for j, b in enumerate(hyp, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (a != b)))
        prev = cur
    return prev[-1] / len(ref)

engines = ["paddle", "manga_ocr", "qwen", "gemini"]
dialog = [it for it in items if it["key"] not in SFX_TRUTH and it["key"] not in SFX_PREFIX and it["key"] not in SKIP]
print(f"對話 {len(dialog)} 框(以 Paddle 為基準)")
for e in engines[1:]:
    scores = [cer(it["paddle"], it.get(e, "")) for it in dialog]
    wrong = sum(s > 0 for s in scores)
    print(f"  {e:10s} 平均 CER {sum(scores)/len(scores):6.2%}   有錯的框 {wrong:3d}")

print(f"\n狀聲詞 {len(SFX_TRUTH) + len(SFX_PREFIX)} 框")
for e in engines:
    ok = sum(norm(it.get(e, "")) == norm(SFX_TRUTH[it["key"]]) for it in items if it["key"] in SFX_TRUTH)
    ok += sum(norm(it.get(e, "")).startswith(SFX_PREFIX[it["key"]]) for it in items if it["key"] in SFX_PREFIX)
    print(f"  {e:10s} 答對 {ok}")

## 6. Gemini 和 Paddle 不一致的框

In [ ]:
import html, base64
from IPython.display import HTML, display
diff = [it for it in dialog if cer(it["paddle"], it["gemini"]) > 0]
diff += [it for it in items if it["key"] in SFX_TRUTH or it["key"] in SFX_PREFIX]
rows = "".join(
    f"<tr><td>{it['key']}</td><td><img src='data:image/png;base64,{base64.b64encode(open(f'{DRIVE_DIR}/{it['crop']}', 'rb').read()).decode()}' style='max-height:180px'></td>"
    + "".join(f"<td>{html.escape(it.get(e, ''))}</td>" for e in engines) + "</tr>"
    for it in diff)
page = ("<meta charset=utf-8><style>td,th{border:1px solid #ccc;padding:4px;vertical-align:top}table{border-collapse:collapse}</style>"
        "<table><tr><th>框</th><th>裁圖</th>" + "".join(f"<th>{e}</th>" for e in engines) + "</tr>" + rows + "</table>")
open(f"{DRIVE_DIR}/report_gemini.html", "w").write(page)
display(HTML(page))
print(len(diff), "列;已存 report_gemini.html")